In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ================== Dataset and Model Classes (same as original) ==================

class FruitsDataset:
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

class FrequencyDomainDataset:
    """Custom dataset for frequency domain representations"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase

class FrequencyDomainCNN(nn.Module):
    """Enhanced ResNet50-based model for frequency domain"""
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(FrequencyDomainCNN, self).__init__()
        
        self.model = models.resnet50(pretrained=False)
        self.model.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.model(x)

def spatial_to_frequency(images):
    """Convert spatial domain images to frequency domain using FFT"""
    freq_complex = torch.fft.fft2(images, dim=(-2, -1))
    freq_complex = torch.fft.fftshift(freq_complex, dim=(-2, -1))
    
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    return freq_features, freq_phase, freq_complex

# ================== Evaluation and Visualization Functions ==================

def evaluate_model(model, test_loader, classes):
    """Evaluate model and collect predictions"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    print("\nEvaluating model on test set...")
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images = freq_images.to(device)
            outputs = model(freq_images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)
    
    return all_predictions, all_labels, all_probabilities

def plot_confusion_matrix(y_true, y_pred, classes, figsize=(20, 18), save_path=None):
    """Plot confusion matrix with enhanced visualization"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate percentages
    cm_percentage = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Raw counts
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count'}, ax=ax1)
    ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax1.set_title('Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
    ax1.tick_params(axis='x', rotation=90, labelsize=8)
    ax1.tick_params(axis='y', rotation=0, labelsize=8)
    
    # Plot 2: Percentages
    sns.heatmap(cm_percentage, annot=False, fmt='.1f', cmap='YlOrRd',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Percentage (%)'}, ax=ax2)
    ax2.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax2.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax2.set_title('Confusion Matrix (Percentage)', fontsize=14, fontweight='bold')
    ax2.tick_params(axis='x', rotation=90, labelsize=8)
    ax2.tick_params(axis='y', rotation=0, labelsize=8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Confusion matrix saved to: {save_path}")
    
    plt.show()
    
    return cm

def calculate_per_class_metrics(cm, class_idx):
    """Calculate sensitivity, specificity, and error rate for a specific class"""
    tp = cm[class_idx, class_idx]
    fn = np.sum(cm[class_idx, :]) - tp
    fp = np.sum(cm[:, class_idx]) - tp
    tn = np.sum(cm) - tp - fn - fp
    
    # Sensitivity (Recall/True Positive Rate)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # Specificity (True Negative Rate)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # Error Rate
    error_rate = (fp + fn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    
    return sensitivity, specificity, error_rate

def generate_comprehensive_classification_report(y_true, y_pred, classes, save_path=None):
    """Generate comprehensive classification report with all metrics including per-sample details"""
    from sklearn.metrics import cohen_kappa_score
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    
    # Calculate overall metrics
    overall_accuracy = accuracy_score(y_true, y_pred)
    cohen_kappa = cohen_kappa_score(y_true, y_pred)
    
    # Build comprehensive report
    report_lines = []
    report_lines.append("="*100)
    report_lines.append("COMPREHENSIVE CLASSIFICATION REPORT")
    report_lines.append("Frequency Domain CNN - Fruits-360 Dataset")
    report_lines.append("="*100)
    report_lines.append("")
    
    # Overall metrics
    report_lines.append("OVERALL METRICS:")
    report_lines.append("-"*100)
    report_lines.append(f"Overall Accuracy:           {overall_accuracy:.6f} ({overall_accuracy*100:.4f}%)")
    report_lines.append(f"Cohen's Kappa Score:        {cohen_kappa:.6f}")
    report_lines.append(f"Total Test Samples:         {len(y_true)}")
    report_lines.append(f"Number of Classes:          {len(classes)}")
    report_lines.append(f"Correctly Classified:       {np.sum(y_true == y_pred)}")
    report_lines.append(f"Misclassified:              {np.sum(y_true != y_pred)}")
    report_lines.append("")
    
    # Per-class metrics header
    report_lines.append("="*100)
    report_lines.append("PER-CLASS METRICS:")
    report_lines.append("="*100)
    report_lines.append("")
    report_lines.append(f"{'Class Name':<30} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Sensitivity':<13} {'Specificity':<13} {'Error Rate':<12} {'Support':<10}")
    report_lines.append("-"*100)
    
    # Calculate and display per-class metrics
    for i, class_name in enumerate(classes):
        sensitivity, specificity, error_rate = calculate_per_class_metrics(cm, i)
        
        report_lines.append(
            f"{class_name:<30} "
            f"{precision[i]:<12.6f} "
            f"{recall[i]:<12.6f} "
            f"{f1[i]:<12.6f} "
            f"{sensitivity:<13.6f} "
            f"{specificity:<13.6f} "
            f"{error_rate:<12.6f} "
            f"{support[i]:<10}"
        )
    
    report_lines.append("-"*100)
    
    # Weighted averages
    weighted_precision = np.average(precision, weights=support)
    weighted_recall = np.average(recall, weights=support)
    weighted_f1 = np.average(f1, weights=support)
    
    # Calculate weighted sensitivity, specificity, and error rate
    weighted_sensitivity = 0
    weighted_specificity = 0
    weighted_error_rate = 0
    total_support = np.sum(support)
    
    for i in range(len(classes)):
        sensitivity, specificity, error_rate = calculate_per_class_metrics(cm, i)
        weighted_sensitivity += sensitivity * support[i] / total_support
        weighted_specificity += specificity * support[i] / total_support
        weighted_error_rate += error_rate * support[i] / total_support
    
    report_lines.append(
        f"{'WEIGHTED AVERAGE':<30} "
        f"{weighted_precision:<12.6f} "
        f"{weighted_recall:<12.6f} "
        f"{weighted_f1:<12.6f} "
        f"{weighted_sensitivity:<13.6f} "
        f"{weighted_specificity:<13.6f} "
        f"{weighted_error_rate:<12.6f} "
        f"{total_support:<10}"
    )
    
    # Macro averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    
    macro_sensitivity = np.mean([calculate_per_class_metrics(cm, i)[0] for i in range(len(classes))])
    macro_specificity = np.mean([calculate_per_class_metrics(cm, i)[1] for i in range(len(classes))])
    macro_error_rate = np.mean([calculate_per_class_metrics(cm, i)[2] for i in range(len(classes))])
    
    report_lines.append(
        f"{'MACRO AVERAGE':<30} "
        f"{macro_precision:<12.6f} "
        f"{macro_recall:<12.6f} "
        f"{macro_f1:<12.6f} "
        f"{macro_sensitivity:<13.6f} "
        f"{macro_specificity:<13.6f} "
        f"{macro_error_rate:<12.6f} "
        f"{total_support:<10}"
    )
    
    report_lines.append("-"*100)
    report_lines.append("")
    
    # Per-sample predictions
    report_lines.append("="*100)
    report_lines.append("PER-SAMPLE PREDICTIONS:")
    report_lines.append("="*100)
    report_lines.append("")
    report_lines.append(f"{'Sample #':<10} {'True Label':<30} {'Predicted Label':<30} {'Result':<15}")
    report_lines.append("-"*100)
    
    for idx in range(len(y_true)):
        true_label = classes[y_true[idx]]
        pred_label = classes[y_pred[idx]]
        result = "CORRECT ✓" if y_true[idx] == y_pred[idx] else "INCORRECT ✗"
        
        report_lines.append(
            f"{idx+1:<10} "
            f"{true_label:<30} "
            f"{pred_label:<30} "
            f"{result:<15}"
        )
    
    report_lines.append("-"*100)
    report_lines.append("")
    report_lines.append("="*100)
    report_lines.append("END OF REPORT")
    report_lines.append("="*100)
    
    # Print to console
    report_text = "\n".join(report_lines)
    print(report_text)
    
    # Save to file
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        print(f"\nComprehensive classification report saved to: {save_path}")
    
    return report_text

def generate_summary_metrics(y_true, y_pred, classes):
    """Generate summary statistics"""
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    
    print("\n" + "="*80)
    print("SUMMARY METRICS")
    print("="*80)
    print(f"Overall Accuracy:        {accuracy*100:.2f}%")
    print(f"Weighted Precision:      {precision:.4f}")
    print(f"Weighted Recall:         {recall:.4f}")
    print(f"Weighted F1-Score:       {f1:.4f}")
    print(f"Total Test Samples:      {len(y_true)}")
    print(f"Number of Classes:       {len(classes)}")
    print("="*80)

# ================== Main Execution ==================

def main():
    print("="*80)
    print("CONFUSION MATRIX & CLASSIFICATION REPORT GENERATOR")
    print("Frequency Domain CNN - Fruits-360 Dataset")
    print("="*80)
    
    # MODIFY THESE PATHS
    data_root = r'C:\Users\CSE_SDPL\Downloads\data\fruits-360_100x100\fruits-360'
    model_path = 'fruits_fdcnn_fixed_scorecam.pth'
    
    # Check paths
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        return
    
    if not os.path.exists(model_path):
        print(f"\nERROR: Model file not found: {model_path}")
        print("Please train the model first using the main script.")
        return
    
    # Load test dataset
    print("\n[Step 1] Loading test dataset...")
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_dir = os.path.join(data_root, 'Test')
    testset = FruitsDataset(test_dir, transform=transform_test)
    classes = testset.classes
    
    print(f"Number of classes: {len(classes)}")
    print(f"Test samples: {len(testset)}")
    
    # Convert to frequency domain
    print("\n[Step 2] Converting to frequency domain...")
    freq_test_dataset = FrequencyDomainDataset(testset)
    test_loader = DataLoader(freq_test_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Load model
    print("\n[Step 3] Loading trained model...")
    checkpoint = torch.load(model_path, map_location=device)
    
    model = FrequencyDomainCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"Model loaded successfully!")
    print(f"Saved test accuracy: {checkpoint.get('test_accuracy', 'N/A'):.2f}%")
    
    # Evaluate model
    print("\n[Step 4] Evaluating model and collecting predictions...")
    y_pred, y_true, y_proba = evaluate_model(model, test_loader, classes)
    
    # Generate summary metrics
    print("\n[Step 5] Generating summary metrics...")
    generate_summary_metrics(y_true, y_pred, classes)
    
    # Generate comprehensive classification report with all metrics
    print("\n[Step 6] Generating comprehensive classification report...")
    print("(This may take a moment for large datasets...)")
    generate_comprehensive_classification_report(y_true, y_pred, classes, save_path='classification_report.txt')
    
    # Generate confusion matrix
    print("\n[Step 7] Generating confusion matrix...")
    cm = plot_confusion_matrix(y_true, y_pred, classes, 
                               figsize=(20, 18), 
                               save_path='confusion_matrix.png')
    
    print("\n" + "="*80)
    print("EVALUATION COMPLETED SUCCESSFULLY!")
    print("="*80)
    print("\nGenerated files:")
    print("  ✓ classification_report.txt  - Comprehensive report with all metrics and per-sample predictions")
    print("  ✓ confusion_matrix.png       - Visual confusion matrix (counts and percentages)")
    print("="*80)
    print("\nThe classification_report.txt contains:")
    print("  • Overall Accuracy & Cohen's Kappa Score")
    print("  • Per-Class Metrics: Precision, Recall, F1-Score, Sensitivity, Specificity, Error Rate, Support")
    print("  • Weighted & Macro Averages")
    print("  • Per-Sample Predictions (True Label, Predicted Label, Result)")
    print("="*80)

if __name__ == "__main__":
    main()